### Installation

In [1]:
 %%capture
!pip install unsloth vllm
# Temporarily install a specific TRL nightly version
!pip install git+https://github.com/huggingface/trl.git@e95f9fb74a3c3647b86f251b7e230ec51c64b72b
!pip install triton==3.1.0
!pip install -U pynvml

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [2]:
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported
import torch
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Load up `Llama 3.1 8B Instruct`, and set parameters

In [3]:
# Optional: patching if you plan to experiment with RL in the future (not used here)
# PatchFastRL("GRPO", FastLanguageModel)

# Model and training configuration
max_seq_length = 512  # Adjust this if you need longer sequences
lora_rank = 32        # LoRA rank for efficient fine-tuning

# Load the base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/meta-Llama-3.1-8B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,         # Use 4-bit quantization to reduce memory usage
    fast_inference=True,       # Enable vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,
)



INFO 02-21 19:06:07 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.59%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 2.61 GB. Also swap space = 5 GB.
WARNING 02-21 19:06:09 config.py:2448] Casting torch.bfloat16 to torch.float16.
INFO 02-21 19:06:22 config.py:549] This model

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Capturing CUDA graph shapes: 100%|██████████| 23/23 [00:36<00:00,  1.57s/it]

INFO 02-21 19:07:58 model_runner.py:1562] Graph capturing finished in 36 secs, took 0.59 GiB
INFO 02-21 19:07:58 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 64.25 seconds


tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [4]:
# Apply LoRA for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # LoRA rank, suggested values: 8, 16, 32, 64, 128
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Enables memory-efficient training for long sequences
    random_state=3407,
)

Unsloth 2025.2.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### Data Prep


Using the start wars dataset


In [7]:
import os
os.getcwd()
file_path = "/kaggle/input/starwars-data/SW_EpisodeIV_VI.json"

In [8]:
# import os
from datasets import load_dataset

# file_path = "/data/SW_EpisodeIV_VI.json"

# # Check if the file exists
# if not os.path.exists(file_path):
#     print(f"Error: The file '{file_path}' was not found. Please ensure the file exists at the specified path.")
#     exit() # Exit if file doesn't exist

dataset = load_dataset("json", data_files={"train": file_path})["train"]

# Preprocessing: Combine the character and line into one string.
def preprocess(example):
    # Create a dialogue-like string (e.g., "THREEPIO: We're doomed!")
    text = f"{example['Character']}: {example['Line']}"
    # Tokenize the text; we use padding and truncation to ensure uniform length.
    tokenized = tokenizer(text, truncation=True, max_length=max_seq_length, padding="max_length")
    # For language modeling, labels are the same as the input IDs.
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Apply the preprocessing to the entire dataset.
tokenized_dataset = dataset.map(preprocess, batched=False)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2523 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [9]:
# Apply the preprocessing to the entire dataset.
tokenized_dataset = dataset.map(preprocess, batched=False)

# Define training arguments using Hugging Face's Trainer API.
training_args = TrainingArguments(
    output_dir="./star_wars_model",         # Directory to save model checkpoints
    per_device_train_batch_size=1,            # Adjust based on your GPU memory
    gradient_accumulation_steps=1,            # Increase for smoother training if needed
    num_train_epochs=2,                       # Set epochs based on your dataset size
    learning_rate=5e-6,                       # Adjust the learning rate as needed
    weight_decay=0.1,
    logging_steps=10,                         # Log every 10 steps
    save_steps=500,                           # Save a checkpoint every 500 steps
    fp16=not is_bfloat16_supported(),         # Use FP16 if BF16 is not supported
    bf16=is_bfloat16_supported(),
    report_to = "none"
)



Map:   0%|          | 0/2523 [00:00<?, ? examples/s]

In [10]:
# Initialize the Trainer; this automatically uses cross-entropy loss for language modeling.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

# Start training
trainer.train()

<ipython-input-10-f7a520dc5ae8>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 2,523 | Num Epochs = 2
O^O/ \_/ \    Batch size per device = 1 | Gradient Accumulation steps = 1
\        /    Total batch size = 1 | Total steps = 5,046
 "-____-"     Number of trainable parameters = 83,886,080


Step,Training Loss
10,14.976400
20,14.191600
30,13.151300
40,13.133200
50,11.386200
60,12.275800
70,11.149700
80,11.267200
90,10.256100
100,9.012500


TrainOutput(global_step=5046, training_loss=5.962414267891889, metrics={'train_runtime': 10225.2198, 'train_samples_per_second': 0.493, 'train_steps_per_second': 0.493, 'total_flos': 6.49075618700329e+16, 'train_loss': 5.962414267891889, 'epoch': 2.0})

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [60]:
text = tokenizer.apply_chat_template([
    {"role": "user", "content": "Speak like a wise Jedi about the nature of the Force."},
], tokenize=False, add_generation_prompt=True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:22<00:00, 22.27s/it, est. speed input: 2.16 toks/s, output: 19.94 toks/s]


'(The wise Jedi Master takes a deep breath, closing his eyes as he focuses on the vast expanse of the Force)\n\n"Ah, the Force. It surrounds us, it binds us, it connects us to the universe in ways both seen and unseen. A powerful, yet subtle energy that flows through all living things, it is the essence of life itself. Through the Force, we are one with the stars, the trees, the rivers, and the creatures of the galaxy.\n\n"In its depths, we find balance and harmony. The dark and the light, the calm and the turbulent, all exist in eternal dance. The Force is not a tool to be wielded, but a guiding principle that shapes our actions and decisions. It whispers wisdom in the silence, a gentle breeze that rustles the leaves of our souls.\n\n"But beware, young one, for the dark side lies in wait. A path of fear, anger, and aggression, it tempts us to succumb to our basest desires. The dark side seeks to dominate, to control, and to destroy. It is a seductive whisper, a siren\'s call that prom

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [63]:
# model.save_lora("kaggle/working/grpo_saved_lora")

!zip -r train_token.zip /kaggle/working/star_wars_model_finetuned

  adding: kaggle/working/star_wars_model_finetuned/ (stored 0%)
  adding: kaggle/working/star_wars_model_finetuned/training_args.bin (deflated 51%)
  adding: kaggle/working/star_wars_model_finetuned/tokenizer_config.json (deflated 94%)
  adding: kaggle/working/star_wars_model_finetuned/adapter_config.json (deflated 56%)
  adding: kaggle/working/star_wars_model_finetuned/tokenizer.json (deflated 85%)
  adding: kaggle/working/star_wars_model_finetuned/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/star_wars_model_finetuned/special_tokens_map.json (deflated 71%)
  adding: kaggle/working/star_wars_model_finetuned/README.md (deflated 66%)


In [19]:
# # Save the fine-tuned model
# trainer.save_model("./star_wars_model_finetuned")

# # Optionally, save the tokenizer too for later reloading
# tokenizer.save_pretrained("./star_wars_model_finetuned")

('./star_wars_model_finetuned/tokenizer_config.json',
 './star_wars_model_finetuned/special_tokens_map.json',
 './star_wars_model_finetuned/tokenizer.json')

Now we load the LoRA and test:

In [18]:
# import os

# # Ensure the directory exists
# os.makedirs('kaggle/working', exist_ok=True)

# # Save your LoRA model
# model.save_lora("kaggle/working/grpo_saved_lora")

In [67]:
text = tokenizer.apply_chat_template([
    # {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "speak to like Han Solo"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it, est. speed input: 22.73 toks/s, output: 14.61 toks/s]


"I'm not here to make friends.  I'm only here to do a job, and I'm gonna get it done."

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!